# PyTorch基础

作为深度学习的框架，PyTorch的数据基础便是张量`tensor`类，这一点和Tensorflow、MxNet是一致的。

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
print(f'Import PyTorch V{torch.__version__}')

Import PyTorch V2.1.1


## 创建张量

和其他科学计算库一样，PyTorch提供了很快快捷的创建张量函数，不同的函数生成数据的逻辑不同，但输入的参数一般包括形状、数据类型和存在的设备平台。

主要的创建函数包括：

- `empty`：创建过程不对数据进行初始化，申请的内存空间是什么就是什么；
- `zeros`：数据初始化为0；
- `ones`：数据初始化为1；
- `rand`：数据按照0到1之间的平均分布随机初始化；
- `randn`：数据按照正态分布随机初始化，正态分布的默认均值为0，标准差为1。

In [2]:
dev1 = torch.device('cpu')
dev2 = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for func_name, shape, dtype, dev in [
    ('empty', (2,), torch.double, dev1),
    ('zeros', (1, 3), torch.float32, dev1),
    ('ones', (2, 1, 3), torch.int, dev2),
    ('rand', (2, 2), torch.double, dev2),
    ('randn', (4, 4), torch.float64, dev1),
]:
    func = getattr(torch, func_name)
    t = func(size=shape, dtype=dtype, device=dev)
    print(f'{func_name}, {shape}, {dtype}, {dev}')
    print(t)
    print()

empty, (2,), torch.float64, cpu
tensor([0., 0.], dtype=torch.float64)

zeros, (1, 3), torch.float32, cpu
tensor([[0., 0., 0.]])

ones, (2, 1, 3), torch.int32, cuda
tensor([[[1, 1, 1]],

        [[1, 1, 1]]], device='cuda:0', dtype=torch.int32)

rand, (2, 2), torch.float64, cuda
tensor([[0.2522, 0.7236],
        [0.2732, 0.2509]], device='cuda:0', dtype=torch.float64)

randn, (4, 4), torch.float64, cpu
tensor([[ 0.1810, -0.1355,  0.1346, -0.2900],
        [-0.8761,  0.0222,  0.8057, -0.2702],
        [ 0.6816, -0.8602, -1.0596,  1.3201],
        [ 1.1490,  1.0359,  0.8697, -0.7585]], dtype=torch.float64)



## 转换

PyTorch中的张量可以由数组直接转换而来，也支持与NumPy数组的相互转换。

**注意**：NumPy不支持CUDA平台。

In [3]:
arr = [[0.2, 0.7, 1.7], [-23.0, 81.3, -44e-1]]
t_arr = torch.tensor(arr, dtype=torch.float32, device=dev2)

arr, t_arr

([[0.2, 0.7, 1.7], [-23.0, 81.3, -4.4]],
 tensor([[  0.2000,   0.7000,   1.7000],
         [-23.0000,  81.3000,  -4.4000]], device='cuda:0'))

In [4]:
t_arr.to('cpu').numpy()

array([[  0.2,   0.7,   1.7],
       [-23. ,  81.3,  -4.4]], dtype=float32)

In [5]:
ndarr = np.linspace(0.0, 10.0, 7)
ndarr, torch.from_numpy(ndarr)

(array([ 0.        ,  1.66666667,  3.33333333,  5.        ,  6.66666667,
         8.33333333, 10.        ]),
 tensor([ 0.0000,  1.6667,  3.3333,  5.0000,  6.6667,  8.3333, 10.0000],
        dtype=torch.float64))

## 简单运算

主要包括三类：

- 运算符的重载，如`+`、`-`、`*`等；
- 运算函数，例如`add`、`dot`、`mv`、`mm`等；
- 自运算函数，例如`add_`，在运算函数后加下划线，运算的第一个变量和最终结果的保留都是自身。

In [6]:
t0, t1, v0, v1, m0, m1 = \
    torch.rand(2, 3), torch.randn(2, 3), \
    torch.rand(4), torch.randn(4), \
    torch.rand(4, 4), torch.randn(4, 4)

In [7]:
t0 + t1, t0 / t1 + t0 * t1

(tensor([[ 1.3698, -0.0649,  3.5456],
         [ 1.1926,  0.8537,  2.3325]]),
 tensor([[ 2.0770, -0.8461,  1.8191],
         [ 3.0459,  1.3143,  2.0117]]))

In [8]:
torch.add(t0, t1), torch.add(torch.divide(t0, t1), torch.multiply(t0, t1))

(tensor([[ 1.3698, -0.0649,  3.5456],
         [ 1.1926,  0.8537,  2.3325]]),
 tensor([[ 2.0770, -0.8461,  1.8191],
         [ 3.0459,  1.3143,  2.0117]]))

In [9]:
v0 * v1, torch.dot(v0, v1)

(tensor([ 0.1297,  1.0664, -0.3389, -0.2764]), tensor(0.5808))

In [10]:
m0 * v0, torch.mv(m0, v0)

(tensor([[0.1108, 0.4570, 0.0008, 0.3520],
         [0.3023, 0.7954, 0.0975, 0.0052],
         [0.0424, 0.7499, 0.0483, 0.4054],
         [0.2881, 0.1939, 0.1659, 0.0541]]),
 tensor([0.9206, 1.2004, 1.2459, 0.7019]))

In [11]:
m0 * m1, torch.mm(m0, m1)

(tensor([[-0.4847, -0.1179, -0.0058,  0.4256],
         [ 0.3466, -0.4332, -0.1427,  0.0030],
         [-0.1261, -1.0536, -0.3314, -0.4525],
         [-0.4644, -0.2322,  0.7561, -0.0726]]),
 tensor([[-0.6366, -1.0067, -0.0467, -0.0924],
         [-1.5112, -1.3062, -2.0998,  0.5507],
         [-0.5206, -1.5432, -0.1097, -0.3527],
         [-2.1320, -1.5244, -2.2549,  0.0128]]))

In [12]:
t0.add_(t1)
t0

tensor([[ 1.3698, -0.0649,  3.5456],
        [ 1.1926,  0.8537,  2.3325]])

In [17]:
print('Use @ operator')
print(f'm0 @ v0 = {m0 @ v0}')
print(f'm0 @ m1 = {m0 @ m1}')

Use @ operator
m0 @ v0 = tensor([0.9206, 1.2004, 1.2459, 0.7019])
m0 @ m1 = tensor([[-0.6366, -1.0067, -0.0467, -0.0924],
        [-1.5112, -1.3062, -2.0998,  0.5507],
        [-0.5206, -1.5432, -0.1097, -0.3527],
        [-2.1320, -1.5244, -2.2549,  0.0128]])


## 变换形状

PyTorch继承了NumPy中变换数组形状的函数`reshape`，同时还提供了另一个变换形状的函数`view`。

In [13]:
t0.reshape(3, 2), t0.view(3, 2)

(tensor([[ 1.3698, -0.0649],
         [ 3.5456,  1.1926],
         [ 0.8537,  2.3325]]),
 tensor([[ 1.3698, -0.0649],
         [ 3.5456,  1.1926],
         [ 0.8537,  2.3325]]))

In [14]:
m0.reshape(-1, 8), m0.view(-1, 8)

(tensor([[0.3211, 0.5043, 0.0043, 0.6241, 0.8761, 0.8777, 0.5170, 0.0092],
         [0.1228, 0.8275, 0.2562, 0.7186, 0.8350, 0.2139, 0.8794, 0.0958]]),
 tensor([[0.3211, 0.5043, 0.0043, 0.6241, 0.8761, 0.8777, 0.5170, 0.0092],
         [0.1228, 0.8275, 0.2562, 0.7186, 0.8350, 0.2139, 0.8794, 0.0958]]))

In [15]:
m1.view(8, -1)

tensor([[-1.5095, -0.2337],
        [-1.3659,  0.6820],
        [ 0.3956, -0.4935],
        [-0.2760,  0.3254],
        [-1.0267, -1.2731],
        [-1.2937, -0.6297],
        [-0.5561, -1.0854],
        [ 0.8597, -0.7577]])

`view`除了可以用来改变形状，还可以改变数据类型。

In [16]:
m1.view(torch.double)

tensor([[-5.8350e-08,  2.3345e-04],
        [-2.7362e-05,  6.7432e-07],
        [-3.7037e-02, -1.3122e-04],
        [-1.3150e-02, -5.4855e-04]], dtype=torch.float64)